In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

sessions = pd.read_csv('data/sessions.csv')
events = pd.read_csv('data/events.csv')
users = pd.read_csv('data/users.csv')
orders = pd.read_csv('data/orders.csv')
order_items = pd.read_csv('data/order_items.csv')
products = pd.read_json('data/products.json')

print("Sessions shape:", sessions.shape)
print(sessions.head(3))
print("\nVariants:", sessions['variant'].value_counts(dropna=False))


In [2]:
# Data Load
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

sessions = pd.read_csv('data/sessions.csv')
events = pd.read_csv('data/events.csv')
users = pd.read_csv('data/users.csv')
orders = pd.read_csv('data/orders.csv')
order_items = pd.read_csv('data/order_items.csv')
products = pd.read_json('data/products.json')

print("Loaded")
print("Sessions shape:", sessions.shape)
print("Variants:\n", sessions['variant'].value_counts(dropna=False))
print("\nOrders shape:", orders.shape)


Loaded
Sessions shape: (9036, 8)
Variants:
 variant
NaN    6027
A      1527
B      1482
Name: count, dtype: int64

Orders shape: (707, 9)


In [3]:
# CLEANING
sessions['channel'] = sessions['channel'].str.lower().str.strip()
sessions['device'] = sessions['device'].fillna('unknown')
sessions = sessions.drop_duplicates(subset=['session_id'])
print("Clean sessions:", sessions.shape)

events['event_ts'] = pd.to_datetime(events['event_ts'])
print("Clean events:", events.shape)

# Orders outliers (cap 99th percentile)
orders['gross_amount'] = pd.to_numeric(orders['gross_amount'], errors='coerce').fillna(0)
orders['net_amount'] = pd.to_numeric(orders['net_amount'], errors='coerce').fillna(0)
q99 = orders['net_amount'].quantile(0.99)
orders['revenue_cap'] = orders['net_amount'].clip(upper=q99)
orders['is_outlier'] = orders['net_amount'] > q99
print(f"Clean orders: {orders.shape}, Outliers: {orders['is_outlier'].sum()}")

print("Base ETL ready")

Clean sessions: (9000, 8)
Clean events: (18945, 3)
Clean orders: (707, 11), Outliers: 8
Base ETL ready


In [6]:
# Output 1: fact_sessions.csv (matches spec exactly)
funnel_steps = ['product_view', 'add_to_cart', 'begin_checkout', 'payment_attempt', 'purchase']

# Funnel flags (vectorized - fast)
event_flags = events[events['event_type'].isin(funnel_steps)].groupby('session_id')['event_type'].value_counts().unstack(fill_value=0)
funnel_flags = pd.DataFrame(index=sessions['session_id'].unique())
for step in funnel_steps:
    flag_col = f'has_{step.replace("_", "")}'
    funnel_flags[flag_col] = event_flags[step] > 0 if step in event_flags else False

funnel_flags = funnel_flags.reset_index().fillna({'has_productview':False, 'has_addtocart':False, 'has_begincheckout':False, 'has_paymentattempt':False, 'has_purchase':False})

# Session duration (first/last event per session)
event_times = events.groupby('session_id')['event_ts'].agg(['min', 'max']).reset_index()
event_times['session_duration_sec'] = (event_times['max'] - event_times['min']).dt.total_seconds()
event_times = event_times[['session_id', 'session_duration_sec']].fillna(0)

# Time to steps (first occurrence)
def first_step_time(group, step):
    step_time = group[group['event_type']==step]['event_ts'].min()
    if pd.isna(step_time): return np.nan
    return (step_time - group['event_ts'].min()).total_seconds()

step_times = {}
for step in ['add_to_cart', 'begin_checkout', 'purchase']:
    step_times[f'time_to_{step.replace("_","")}_sec']


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_7220\777218082.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  funnel_flags = funnel_flags.reset_index().fillna({'has_productview':False, 'has_addtocart':False, 'has_begincheckout':False, 'has_paymentattempt':False, 'has_purchase':False})


KeyError: 'time_to_addtocart_sec'

In [7]:
# Output 1: fact_sessions.csv
funnel_steps = ['product_view', 'add_to_cart', 'begin_checkout', 'payment_attempt', 'purchase']

# Funnel flags
funnel_flags_dict = {f'has_{s.replace("_","")}': False for s in funnel_steps}
funnel_df = pd.DataFrame([funnel_flags_dict] * len(sessions), index=sessions.index)

# Set True where events exist
step_counts = events[events['event_type'].isin(funnel_steps)].groupby(['session_id', 'event_type']).size().unstack(fill_value=0)
for step in funnel_steps:
    col = f'has_{step.replace("_","")}'
    mask = step_counts.index.isin(sessions['session_id'])
    session_idx = sessions[sessions['session_id'].isin(step_counts.index[mask])].index
    funnel_df.loc[session_idx, col] = step_counts.loc[mask, step] > 0

# Timings (session level - fast)
event_agg = events.groupby('session_id')['event_ts'].agg(['first', 'last', 'min'])
event_agg['session_duration_sec'] = (event_agg['last'] - event_agg['first']).dt.total_seconds()

# First step times
step_first = events[events['event_type'].isin(['add_to_cart', 'begin_checkout', 'purchase'])].groupby(['session_id', 'event_type'])['event_ts'].first().unstack()
for step in ['add_to_cart', 'begin_checkout', 'purchase']:
    col = f'time_to_{step.replace("_","")}_sec'
    if step in step_first:
        step_df = step_first[step].reset_index(name='step_time')
        step_df['time_diff'] = (step_df['step_time'] - events.groupby('session_id')['event_ts'].first().reindex(step_df['session_id']).values).dt.total_seconds()
        event_agg[col] = step_df.set_index('session_id')['time_diff']

event_agg = event_agg.reset_index().fillna(0)

# Merge all
fact_sessions = sessions.reset_index(drop=True).merge(funnel_df.reset_index(drop=True), left_index=True, right_index=True, how='left').merge(event_agg, on='session_id', how='left')

# Revenue
session_rev = orders.groupby('session_id')['net_amount'].sum().reset_index(name='net_amount')
fact_sessions = fact_sessions.merge(session_rev, on='session_id', how='left').fillna({'net_amount':0})

# Save
fact_sessions.to_csv('fact_sessions.csv', index=False)
print("fact_sessions.csv SAVED!")
print("Shape:", fact_sessions.shape)
print("\nColumns:", list(fact_sessions.columns))
print("\nSample:")
print(fact_sessions[['session_id', 'variant', 'has_purchase', 'session_duration_sec',
            'net_amount'
        ]
    ].head()
)


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_7220\2359738461.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan nan ... nan nan nan]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  funnel_df.loc[session_idx, col] = step_counts.loc[mask, step] > 0


fact_sessions.csv SAVED!
Shape: (9000, 21)

Columns: ['session_id', 'user_id', 'session_start_ts', 'device', 'channel', 'campaign_id', 'variant', 'is_new_user', 'has_productview', 'has_addtocart', 'has_begincheckout', 'has_paymentattempt', 'has_purchase', 'first', 'last', 'min', 'session_duration_sec', 'time_to_addtocart_sec', 'time_to_begincheckout_sec', 'time_to_purchase_sec', 'net_amount']

Sample:
  session_id variant has_purchase  session_duration_sec  net_amount
0   S0000001       A        False                   0.0         0.0
1   S0000002     NaN        False                   0.0         0.0
2   S0000003     NaN          NaN                  23.0         0.0
3   S0000004     NaN        False                   0.0         0.0
4   S0000005     NaN          NaN                 100.0         0.0


In [8]:
# Fix bool columns
bool_cols = ['has_productview', 'has_addtocart', 'has_begincheckout', 'has_paymentattempt', 'has_purchase']
for col in bool_cols:
    if col in fact_sessions.columns:
        fact_sessions[col] = fact_sessions[col].astype(bool)
print("Bool columns fixed")
fact_sessions.to_csv('fact_sessions.csv', index=False)  # Re-save


Bool columns fixed


In [10]:
# Output 2: fact_orders.csv
import json
# 1) Basket line revenue
order_items['total_price'] = order_items['quantity'] * order_items['unit_price']

# 2) Join products for category, cost, rating
order_items_enriched = order_items.merge(
    products[['product_id', 'category', 'cost', 'rating']],
    on='product_id',
    how='left'
)

# 3) Per-order basket aggregates
order_summary = order_items_enriched.groupby('order_id').agg(
    total_items=('quantity', 'sum'),
    distinct_products=('product_id', 'nunique'),
    top_category=('category', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else None),
    avg_rating=('rating', 'mean'),
    total_cost=('cost', lambda c: np.sum(c * order_items_enriched.loc[c.index, 'quantity'])),
).reset_index()

# 4) Category mix as JSON (share of each category in basket)
def category_mix_json(df):
    counts = df['category'].value_counts(normalize=True).round(3)
    return json.dumps(counts.to_dict())

cat_mix = order_items_enriched.groupby('order_id').apply(category_mix_json).reset_index(name='category_mix')

# 5) Merge with orders table
fact_orders = orders.merge(order_summary, on='order_id', how='left').merge(cat_mix, on='order_id', how='left')

# Margin proxy: net_amount - total_cost
fact_orders['margin_proxy'] = fact_orders['net_amount'] - fact_orders['total_cost']

# Reorder / select columns per spec
fact_orders = fact_orders[[
    'order_id',
    'session_id',
    'user_id',
    'order_ts',
    'payment_method',
    'net_amount',
    'gross_amount',
    'discount_amount',
    'total_items',
    'distinct_products',
    'top_category',
    'category_mix',
    'avg_rating',
    'total_cost',
    'margin_proxy'
]]

fact_orders.to_csv('fact_orders.csv', index=False)
print("fact_orders.csv saved:", fact_orders.shape)
print(fact_orders.head(3).to_string())


fact_orders.csv saved: (707, 15)
   order_id session_id  user_id             order_ts payment_method  net_amount  gross_amount  discount_amount  total_items  distinct_products top_category                                         category_mix  avg_rating  total_cost  margin_proxy
0  O0000001   S0000015  U000739  2026-01-12T14:51:51            UPI     4095.09       4145.06            49.97            5                  3        Books     {"Sports": 0.333, "Home": 0.333, "Books": 0.333}    4.233333     2580.45       1514.64
1  O0000002   S0000060      NaN  2025-11-19T07:56:21     NetBanking     4151.41       4302.50           151.09            5                  3        Books  {"Books": 0.333, "Sports": 0.333, "Fashion": 0.333}    3.900000     2819.76       1331.65
2  O0000003   S0000073      NaN  2025-12-08T05:40:42     NetBanking     6602.83       7199.78           596.95            2                  1        Books                                       {"Books": 1.0}    4.100000     4

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_7220\2454832494.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cat_mix = order_items_enriched.groupby('order_id').apply(category_mix_json).reset_index(name='category_mix')


In [13]:
# Output 3: dim_users_enriched.csv - FIXED
user_sessions = sessions.groupby('user_id')['session_id'].count().reset_index(name='lifetime_sessions')
user_orders = orders.groupby('user_id')['order_id'].count().reset_index(name='lifetime_orders')

user_first_last = orders.groupby('user_id')['order_ts'].agg(['first', 'last']).reset_index()
user_first_last.columns = ['user_id', 'first_order_date', 'last_order_date']

user_rev = orders.groupby('user_id')['net_amount'].sum().reset_index(name='lifetime_revenue')

# Merge with USERS (your cleaned df)
dim_users = users.merge(user_sessions, how='left', on='user_id').merge(
    user_orders, how='left', on='user_id').merge(
    user_first_last, how='left', on='user_id').merge(
    user_rev, how='left', on='user_id')

# Fill NaNs
dim_users = dim_users.fillna({
    'lifetime_sessions': 1,
    'lifetime_orders': 0,
    'first_order_date': None,
    'last_order_date': None,
    'lifetime_revenue': 0
})

# Flags
dim_users['repeat_rate'] = dim_users['lifetime_orders'] >= 2
dim_users['user_value_band'] = pd.cut(dim_users['lifetime_revenue'],
    bins=[0, 500, 2000, float('inf')], labels=['low', 'medium', 'high'])

# Final columns
cols = ['user_id', 'signup_date', 'city_tier', 'segment', 'preferred_device',
        'lifetime_sessions', 'lifetime_orders', 'first_order_date', 
        'last_order_date', 'repeat_rate', 'lifetime_revenue', 'user_value_band']
dim_users = dim_users[cols]

dim_users.to_csv('dim_users_enriched.csv', index=False)
print("dim_users_enriched.csv saved:", dim_users.shape)
print(dim_users[['user_id', 'lifetime_orders', 'repeat_rate', 'user_value_band']].head())
print("\nValue bands:", dim_users['user_value_band'].value_counts())


ValueError: Must specify a fill 'value' or 'method'.

In [14]:
# Output 3: dim_users_enriched.csv
user_sessions = sessions.groupby('user_id')['session_id'].count().reset_index(name='lifetime_sessions')
user_orders = orders.groupby('user_id')['order_id'].count().reset_index(name='lifetime_orders')
user_first_last = orders.groupby('user_id')['order_ts'].agg(['first', 'last']).reset_index()
user_first_last.columns = ['user_id', 'first_order_date', 'last_order_date']
user_rev = orders.groupby('user_id')['net_amount'].sum().reset_index(name='lifetime_revenue')

dim_users = users.merge(user_sessions, how='left').merge(user_orders, how='left').merge(
    user_first_last, how='left').merge(user_rev, how='left')

# Fill numeric first
dim_users['lifetime_sessions'] = dim_users['lifetime_sessions'].fillna(1)
dim_users['lifetime_orders'] = dim_users['lifetime_orders'].fillna(0)
dim_users['lifetime_revenue'] = dim_users['lifetime_revenue'].fillna(0)

# Dates stay NaT
dim_users['repeat_rate'] = dim_users['lifetime_orders'] >= 2
dim_users['user_value_band'] = pd.cut(dim_users['lifetime_revenue'], 
    bins=[0, 500, 2000, float('inf')], labels=['low', 'medium', 'high'])

cols = ['user_id', 'signup_date', 'city_tier', 'segment', 'preferred_device',
        'lifetime_sessions', 'lifetime_orders', 'first_order_date', 
        'last_order_date', 'repeat_rate', 'lifetime_revenue', 'user_value_band']
dim_users_final = dim_users[cols].copy()

dim_users_final.to_csv('dim_users_enriched.csv', index=False)
print("dim_users_enriched.csv SAVED:", dim_users_final.shape)
print(dim_users_final.head(3)[['user_id', 'lifetime_orders', 'repeat_rate', 'user_value_band']])
print("\nBands:", dim_users_final['user_value_band'].value_counts())


dim_users_enriched.csv SAVED: (2200, 12)
   user_id  lifetime_orders  repeat_rate user_value_band
0  U000001              0.0        False             NaN
1  U000002              1.0        False            high
2  U000003              0.0        False             NaN

Bands: user_value_band
high      467
medium     44
low         1
Name: count, dtype: int64


In [16]:
# Part C Setup + Funnel Diagnosis
fact_sessions = pd.read_csv('fact_sessions.csv')
fact_orders = pd.read_csv('fact_orders.csv')
dim_users = pd.read_csv('dim_users_enriched.csv')
eligible = fact_sessions[fact_sessions['variant'].notna()].copy()

# Segments (merge dim_users)
eligible_seg = eligible.merge(dim_users[['user_id', 'segment', 'city_tier', 'preferred_device']], on='user_id', how='left')

segments = ['device', 'channel', 'is_new_user', 'segment', 'city_tier']
print("Conv + Rev/Session by Segment (Top diffs):")
for seg in segments:
    if seg in eligible_seg.columns:
        conv_diff = eligible_seg.groupby([seg, 'variant'])['has_purchase'].mean().unstack() * 100
        print(f"\n{seg} Conv %:")
        print(conv_diff.round(1))


Conv + Rev/Session by Segment (Top diffs):

device Conv %:
variant     A      B
device              
unknown  64.3  100.0
web      60.8   63.6

channel Conv %:
variant         A     B
channel                
email        59.4  60.9
organic      57.3  62.7
paid_social  57.4  59.6
referral     62.4  65.8
search       68.6  69.2

is_new_user Conv %:
variant         A     B
is_new_user            
0            60.9  63.1
1            60.9  65.9

segment Conv %:
variant     A     B
segment            
premium  61.7  61.0
regular  61.1  64.5
value    60.4  64.2

city_tier Conv %:
variant       A     B
city_tier            
1          60.0  61.7
2          63.4  65.9
3          57.9  63.1


In [17]:
# C.1 Funnel Diagnosis
funnel_cols = ['has_productview', 'has_addtocart', 'has_begincheckout', 'has_paymentattempt', 'has_purchase']
funnel_table = eligible[funnel_cols + ['variant']].groupby('variant').mean() * 100

print("=== FUNNEL CONVERSION % ===")
print(funnel_table.round(1))
print(f"\nEligible sessions: A={sum(eligible.variant=='A')}, B={sum(eligible.variant=='B')}")

# Drop-offs
drops = funnel_table.diff().iloc[1].abs()
print(f"\nBIGGEST LEAKS:")
print("A biggest drop:", drops.idxmax(), f"{drops.max():.1f}%")
print("B biggest drop:", drops.idxmin(), f"{drops.min():.1f}% lift")


=== FUNNEL CONVERSION % ===
         has_productview  has_addtocart  has_begincheckout  \
variant                                                      
A                   60.9           60.9               60.9   
B                   63.9           63.9               63.9   

         has_paymentattempt  has_purchase  
variant                                    
A                      60.9          60.9  
B                      63.9          63.9  

Eligible sessions: A=1523, B=1474

BIGGEST LEAKS:
A biggest drop: has_productview 3.0%
B biggest drop: has_productview 3.0% lift
